In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from scipy import stats
from datasets import load_dataset

# ── Config ────────────────────────────────────────────────────────────────────
LR_START   = 2e-3
LR_END     = 3e-5
N_STEPS    = 5000          # Increased for CIFAR-100
EVAL_EVERY = 200
BATCH_SIZE = 256
SEEDS      = [42, 7, 13, 99, 2025, 1, 8, 21]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load CIFAR-100 ────────────────────────────────────────────────────────────
print('Loading CIFAR-100...')
ds = load_dataset("uoft-cs/cifar100")

train_ds_hf = ds['train']
test_ds_hf  = ds['test']

xtr = np.array([np.array(img) for img in train_ds_hf['img']], dtype=np.uint8)
ytr = np.array(train_ds_hf['fine_label'], dtype=np.int64)

xte = np.array([np.array(img) for img in test_ds_hf['img']], dtype=np.uint8)
yte = np.array(test_ds_hf['fine_label'], dtype=np.int64)

print(f'Train: {xtr.shape} | Test: {xte.shape} | Classes: 100')

# Normalization
CIFAR100_MEAN = np.array([0.5071, 0.4867, 0.4408], dtype=np.float32)
CIFAR100_STD  = np.array([0.2675, 0.2565, 0.2761], dtype=np.float32)

xtr_raw = xtr.astype(np.float32) / 255.0
xte_raw = xte.astype(np.float32) / 255.0

xtr_norm = (xtr_raw - CIFAR100_MEAN) / CIFAR100_STD
xte_norm = (xte_raw - CIFAR100_MEAN) / CIFAR100_STD

xtr_t = torch.tensor(xtr_norm.transpose(0, 3, 1, 2).astype(np.float32))
xte_t = torch.tensor(xte_norm.transpose(0, 3, 1, 2).astype(np.float32))

train_ds = torch.utils.data.TensorDataset(xtr_t, torch.tensor(ytr))
test_ds  = torch.utils.data.TensorDataset(xte_t, torch.tensor(yte))

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

# ── Model ─────────────────────────────────────────────────────────────────────
class FlexCNN(nn.Module):
    def __init__(self, in_ch=3, fc_dim=64*8*8, use_bn1=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(fc_dim, 256)
        self.fc2   = nn.Linear(256, 100)          # 100 classes
        self.bn1   = nn.BatchNorm2d(32) if use_bn1 else nn.Identity()
        self.bn2   = nn.BatchNorm2d(64)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# ── Cell 2: Pre-Warm setup ────────────────────────────────────────────────────
"""
Pre-Warm on CIFAR-10 — Cell 2: Pre-Warm Setup
─────────────────────────────────────────────
The four params follow the notebook's design rules verbatim:

  patch_size = K              (kernel size)
  n_clusters = F / 2          (half the conv1 filter bank)
  n_patches  = floor((F/4) × (1/d))   (d from Otsu — derived in cell 3)
  scale      = 0.2            (default from sensitivity sweep)

F = 32 (conv1 out_channels). K = 3.
"""

from sklearn.cluster import MiniBatchKMeans

# ── Patch extraction ──────────────────────────────────────────────────────────
def extract_patches(images, patch_size, n_patches):
    B, C, H, W = images.shape
    patches = []
    imgs_np = images.cpu().numpy()
    for b in range(B):
        for _ in range(n_patches):
            i = random.randint(0, H - patch_size)
            j = random.randint(0, W - patch_size)
            patch = imgs_np[b, :, i:i+patch_size, j:j+patch_size]
            patches.append(patch.flatten())
    return np.array(patches)

# ── Inverse Manhattan distance spatial weights ────────────────────────────────
def distance_weights(patch_size):
    center  = patch_size // 2
    weights = []
    for i in range(patch_size):
        for j in range(patch_size):
            dist = abs(i - center) + abs(j - center)
            weights.append(1.0 / (1.0 + dist))
    return np.array(weights)

# ── Conditioner ───────────────────────────────────────────────────────────────
def conditioned_init(model, batch_images, n_clusters, n_patches, scale, patch_size, seed):
    patches = extract_patches(batch_images, patch_size, n_patches)
    patches -= patches.mean(axis=1, keepdims=True)        # encode edges, not brightness

    kmeans    = MiniBatchKMeans(n_clusters=n_clusters, random_state=seed, n_init=3)
    kmeans.fit(patches)
    centroids = kmeans.cluster_centers_.copy()

    dw      = distance_weights(patch_size)
    C_in    = batch_images.shape[1]
    dw_full = np.tile(dw, C_in)                            # [9] → [27] for 3-channel
    centroids = centroids * dw_full[np.newaxis, :]
    centroids = centroids / (np.std(centroids) + 1e-8) * scale

    weight_tensor = torch.tensor(
        centroids.reshape(n_clusters, C_in, patch_size, patch_size),
        dtype=torch.float32
    )

    with torch.no_grad():
        target_weight = model.conv1.weight
        out_channels  = target_weight.shape[0]
        num_to_copy   = min(n_clusters, out_channels)
        target_weight[:num_to_copy].copy_(weight_tensor[:num_to_copy])

    return model

# ── Design-rule params (n_patches derived in cell 3 from Otsu) ────────────────
F_FILTERS = 32          # conv1 output channels
K_KERNEL  = 3           # kernel size

HP_BASE = {
    'n_clusters' : F_FILTERS // 2,    # = 16
    'scale'      : 0.25,
    'patch_size' : K_KERNEL,          # = 3
}
print(f'Pre-Warm base HP: {HP_BASE}')
print('n_patches will be derived from Otsu (cell 3).')

# ── Evaluation function ───────────────────────────────────────────────────────
def evaluate(model, data_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# ── Single-run driver with conditioning + cosine LR + periodic eval ───────────
def run_with_curve(seed, conditioned, hp):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    model     = FlexCNN(in_ch=3, fc_dim=64*8*8).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_START)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_STEPS, eta_min=LR_END
    )

    # Grab first batch for conditioning (operates on normalized 3-ch images)
    first_iter = iter(train_loader)
    images0, _ = next(first_iter)
    if conditioned and hp is not None:
        model = conditioned_init(model, images0, **hp, seed=seed)

    losses, eval_steps, eval_accs = [], [], []
    step = 0
    model.train()
    while step < N_STEPS:
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            losses.append(loss.item())
            step += 1
            if step % EVAL_EVERY == 0 or step == N_STEPS:
                acc = evaluate(model, test_loader)
                eval_steps.append(step)
                eval_accs.append(acc)
                model.train()
            if step >= N_STEPS:
                break

    return {
        'losses'    : np.array(losses),
        'eval_steps': np.array(eval_steps),
        'eval_accs' : np.array(eval_accs),
        'final_loss': losses[-1],
        'final_acc' : eval_accs[-1],
    }

print('Pre-Warm setup ready.')





# ── Cell 3: n_patches using Patch L2 Norm Rule (for color images) ─────────────
print('Computing n_patches using Patch L2 Norm rule...')

def compute_mean_patch_norm(xtr_raw, patch_size=3, n_patches_per_img=20, seed=42):
    rng = np.random.default_rng(seed)
    N_imgs = min(2000, xtr_raw.shape[0])
    img_idx = rng.choice(xtr_raw.shape[0], size=N_imgs, replace=False)

    norms = []
    for idx in img_idx:
        img = xtr_raw[idx]
        for _ in range(n_patches_per_img):
            i = rng.integers(0, 32 - patch_size)
            j = rng.integers(0, 32 - patch_size)
            patch = img[i:i+patch_size, j:j+patch_size, :]
            patch_mean = patch.mean(axis=(0,1), keepdims=True)
            centered = patch - patch_mean
            norm = np.linalg.norm(centered)
            norms.append(norm)
    return float(np.mean(norms))

mean_norm = compute_mean_patch_norm(xtr_raw)

F_FILTERS = 32
C = 6.65

# ── Cell 4: Train on CIFAR-100 with multiple n_patches ───────────────────────
"""
CIFAR-100 — Cell 4: Sweep over n_patches values
Tests several candidate patch counts with 8-seed paired experiments.
"""

THRESHOLDS = [0.25, 0.35, 0.42]   # Reasonable for CIFAR-100

# Test values around your computed n_patches
patches_to_test = [16, 17, 18, 19, 20, 21, 22]

for n_patches in patches_to_test:
    # Important: Create fresh HP for each n_patches value
    HP = {**HP_BASE, 'n_patches': n_patches}

    print(f"\n{'='*60}")
    print(f"Testing n_patches = {n_patches}")
    print(f"HP = {HP}")
    print(f"{'='*60}\n")

    base_runs, cond_runs = [], []

    for i, s in enumerate(SEEDS):
        print(f'Seed {s} ({i+1}/{len(SEEDS)})... ', end='', flush=True)

        br = run_with_curve(s, conditioned=False, hp=None)
        cr = run_with_curve(s, conditioned=True,  hp=HP)

        base_runs.append(br)
        cond_runs.append(cr)

        print(f'base={br["final_acc"]:.4f}  cond={cr["final_acc"]:.4f}  '
              f'Δ={cr["final_acc"]-br["final_acc"]:+.4f}')

    # ── H1: Final Accuracy ───────────────────────────────────────────────────
    base_final_acc  = np.array([r['final_acc']  for r in base_runs])
    cond_final_acc  = np.array([r['final_acc']  for r in cond_runs])
    base_final_loss = np.array([r['final_loss'] for r in base_runs])
    cond_final_loss = np.array([r['final_loss'] for r in cond_runs])

    delta_acc = cond_final_acc.mean() - base_final_acc.mean()
    _, p_acc  = stats.ttest_rel(cond_final_acc, base_final_acc, alternative='greater')
    n_wins    = int((cond_final_acc > base_final_acc).sum())

    print(f'\n── H1 — Final metrics at step {N_STEPS} (n_patches={n_patches}) ──')
    print(f'  Baseline : loss={base_final_loss.mean():.4f}  '
          f'acc={base_final_acc.mean():.4f} ± {base_final_acc.std():.4f}')
    print(f'  Pre-Warm : loss={cond_final_loss.mean():.4f}  '
          f'acc={cond_final_acc.mean():.4f} ± {cond_final_acc.std():.4f}')
    print(f'  Δacc  = {delta_acc:+.4f}   p={p_acc:.4f}   ({n_wins}/{len(SEEDS)} wins)')

    print(f'── Finished n_patches={n_patches} ──\n')

print('All n_patches sweeps completed.')

Device: cuda
Loading CIFAR-100...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: (50000, 32, 32, 3) | Test: (10000, 32, 32, 3) | Classes: 100
Train: 50000 | Test: 10000
Pre-Warm base HP: {'n_clusters': 16, 'scale': 0.25, 'patch_size': 3}
n_patches will be derived from Otsu (cell 3).
Pre-Warm setup ready.
Computing n_patches using Patch L2 Norm rule...

Testing n_patches = 16
HP = {'n_clusters': 16, 'scale': 0.25, 'patch_size': 3, 'n_patches': 16}

Seed 42 (1/8)... base=0.4298  cond=0.4318  Δ=+0.0020
Seed 7 (2/8)... base=0.4283  cond=0.4333  Δ=+0.0050
Seed 13 (3/8)... base=0.4313  cond=0.4350  Δ=+0.0037
Seed 99 (4/8)... base=0.4333  cond=0.4420  Δ=+0.0087
Seed 2025 (5/8)... base=0.4233  cond=0.4352  Δ=+0.0119
Seed 1 (6/8)... base=0.4301  cond=0.4294  Δ=-0.0007
Seed 8 (7/8)... base=0.4245  cond=0.4345  Δ=+0.0100
Seed 21 (8/8)... base=0.4248  cond=0.4382  Δ=+0.0134

── H1 — Final metrics at step 5000 (n_patches=16) ──
  Baseline : loss=0.6694  acc=0.4282 ± 0.0034
  Pre-Warm : loss=0.6786  acc=0.4349 ± 0.0036
  Δacc  = +0.0068   p=0.0033   (7/8 wins)
── Finished